In [1]:
import numpy as np

import sys
sys.path.append("../")

In [2]:
from qiskit import QuantumCircuit
from qitp.hamiltonians.pauli_hamiltonian import PauliHamiltonian
from qitp.engine.propagation import QITEPropagation
from qitp.engine.unitary_approximation import UnitaryApproximation
from qitp.simulation.backend import SimulatorBackend
from qiskit.quantum_info import SparsePauliOp

In [7]:
paulis = ["ZZ", "XI", "IX"]
coeffs = [np.pi/4,np.pi/4,np.pi/4]

H = PauliHamiltonian(paulis, coeffs)

# Define a Pauli basis for the approximation
pauli_basis = [
    SparsePauliOp("YY"),
    SparsePauliOp("YI"),
    SparsePauliOp("IY"),
]

approx = UnitaryApproximation(pauli_basis)

backend = SimulatorBackend()

engine = QITEPropagation(H, approx, backend)

initial_circuit = QuantumCircuit(2)
initial_circuit.h(0)
initial_circuit.h(1)


state = backend.run(initial_circuit)

dt = 0.1
dt_min = 1e-4
dt_max = 0.2
tol = 1e-4
prev_energy = None

for step in range(100):

    state = engine.step(state, dt)

    energy = H.expectation(state).real
    print("Energy:", energy, "dt:", dt)

    if prev_energy is not None:

        delta_E = abs(energy - prev_energy)

        if delta_E < tol:
            print("Converged at step", step)
            break

        delta_E = max(delta_E, 1e-12)

        d = int(np.floor(np.log10(delta_E)))
        dt = 10**(-d)

        dt = min(max(dt, dt_min), dt_max)

    prev_energy = energy

Energy: 1.570602580300682 dt: 0.1
Energy: 1.5700213886125296 dt: 0.1
Energy: 1.5638264701042146 dt: 0.2
Energy: 1.5514610767133863 dt: 0.2
Energy: 1.5329739992431104 dt: 0.2
Energy: 1.508438183157197 dt: 0.2
Energy: 1.4779504407553326 dt: 0.2
Energy: 1.4416310691755374 dt: 0.2
Energy: 1.3996233757312295 dt: 0.2
Energy: 1.3520931124557864 dt: 0.2
Energy: 1.2992278220857638 dt: 0.2
Energy: 1.2412360980633501 dt: 0.2
Energy: 1.1783467614779146 dt: 0.2
Energy: 1.1108079581942305 dt: 0.2
Energy: 1.0388861797298818 dt: 0.2
Energy: 0.9628652117452369 dt: 0.2
Energy: 0.8830450142949828 dt: 0.2
Energy: 0.7997405382594764 dt: 0.2
Energy: 0.7132804826259798 dt: 0.2
Energy: 0.6240059975232487 dt: 0.2
Energy: 0.5322693381269735 dt: 0.2
Energy: 0.43843247474743563 dt: 0.2
Energy: 0.3428656645836316 dt: 0.2
Energy: 0.24594599077937002 dt: 0.2
Energy: 0.14805587454586777 dt: 0.2
Energy: 0.049581566221645376 dt: 0.2
Energy: -0.04908837877637359 dt: 0.2
Energy: -0.14756463309703582 dt: 0.2
Energy: -0.24